# Share an S3 bucket with a partner account using bucket policy and KMS key grant

A partner integration team in another business unit is replacing a nightly CSV email handoff with a direct S3 read pull. The previous architect made the bucket public for one day to unblock them, security caught it, and now you have to design the durable pattern: a single named partner principal, a bucket policy clause scoped to that principal, and a KMS grant that gives them Decrypt without giving them the key. The lab simulates the partner principal as a second IAM role in the same AWS account because asking students to wire two real AWS accounts violates the prerequisite expectation; the policy shapes are identical to the true cross-account form.

**Time.** Plan for about 50 minutes hands-on. Like Lab 02, cleanup schedules the CMK for deletion with a 7-day waiting window.

**Cost.** The in-session spend is fractions of a penny for KMS and S3. The 23-cent residual is the 7-day scheduled-deletion floor on the CMK, identical to Lab 02's price tag. If you run Labs 02 and 04 back to back you accrue both residuals (about 46 cents total) until the windows elapse independently.

This lab maps to AWS SAA-C03 Domain 1: Design Secure Architectures (30% of exam weight).

In [ ]:
# NBVAL_SKIP
# Install labrun-checks. Pinned to a specific version per LAB-CREATION-BLUEPRINT.md.

!pip install --quiet labrun-checks==0.5.0

In [ ]:
# NBVAL_SKIP
# Imports and per-lab constants.

import atexit
import getpass
import json
import time

import boto3
from botocore.exceptions import ClientError

from labrun_checks import (
    register,
    check,
    cleanup,
    run_cleanup,
    CleanupResource,
)

LAB_ID = "cross-account-bucket-sharing"
LAB_TAG_KEY = "labrun:lab-slug"
LAB_TAG_VALUE = LAB_ID
REGION = "us-east-1"

OWNER_ROLE_NAME = f"labrun-{LAB_ID}-owner-role"
PARTNER_ROLE_NAME = f"labrun-{LAB_ID}-partner-role"
CMK_ALIAS = f"alias/labrun-{LAB_ID}"
SEED_KEY = "reports/seed.txt"

BUCKET_NAME = None
CMK_KEY_ID = None
CMK_KEY_ARN = None
GRANT_ID = None

In [ ]:
# NBVAL_SKIP
# Register the session, validate AWS credentials, confirm region.

session_token = getpass.getpass("Paste your session token from labrun.dev: ")
aws_access_key_id = getpass.getpass("AWS_ACCESS_KEY_ID: ")
aws_secret_access_key = getpass.getpass("AWS_SECRET_ACCESS_KEY: ")
aws_session_token = getpass.getpass(
    "AWS_SESSION_TOKEN (leave blank for long-lived credentials): "
).strip()
region_input = input(f"AWS region (default {REGION}): ").strip()
if region_input and region_input != REGION:
    print(f"Region {region_input!r} is not supported for this lab.")
    print(f"SAA Batch 1 labs run in {REGION} (N. Virginia).")
    print("Re-run this cell and accept the default.")
    raise SystemExit(1)

AWS_CREDENTIALS = {
    "aws_access_key_id": aws_access_key_id,
    "aws_secret_access_key": aws_secret_access_key,
    "region": REGION,
}
if aws_session_token:
    AWS_CREDENTIALS["aws_session_token"] = aws_session_token

sts = boto3.client(
    "sts",
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key,
    aws_session_token=aws_session_token or None,
    region_name=REGION,
)
try:
    identity = sts.get_caller_identity()
except ClientError as e:
    print("Credentials are missing or expired. STS GetCallerIdentity failed:")
    print(f"  {e}")
    raise SystemExit(1)

ACCOUNT_ID = identity["Account"]
CALLER_ARN = identity["Arn"]
print(f"Credentials validated. Account: {ACCOUNT_ID}")
print(f"Caller ARN: {CALLER_ARN}")
print(f"Region: {REGION}")

BUCKET_NAME = f"labrun-{LAB_ID}-{ACCOUNT_ID}"
print(f"Bucket name resolved: {BUCKET_NAME}")

register(session_token=session_token, lab_id=LAB_ID, credentials=AWS_CREDENTIALS)
print(f"Session registered for lab_id: {LAB_ID}")

In [ ]:
# NBVAL_SKIP
# Cleanup manifest, atexit safety net, orphan scan.
# The CMK and KMS grant are appended after their IDs are known (Task 1 and
# Task 2). The grant is revoked before the key is scheduled for deletion
# even though grants survive schedule_key_deletion; tidy is tidy.

CLEANUP_MANIFEST: list[CleanupResource] = [
    CleanupResource(
        type="s3_bucket",
        id=BUCKET_NAME,
        region=REGION,
        cli_delete_command=f"aws s3 rb s3://{BUCKET_NAME} --force",
    ),
    CleanupResource(
        type="iam_role",
        id=PARTNER_ROLE_NAME,
        region=REGION,
        cli_delete_command=f"aws iam delete-role --role-name {PARTNER_ROLE_NAME}",
    ),
    CleanupResource(
        type="iam_role",
        id=OWNER_ROLE_NAME,
        region=REGION,
        cli_delete_command=f"aws iam delete-role --role-name {OWNER_ROLE_NAME}",
    ),
    CleanupResource(
        type="kms_alias",
        id=CMK_ALIAS,
        region=REGION,
        cli_delete_command=f"aws kms delete-alias --alias-name {CMK_ALIAS}",
    ),
]
# kms_grant and kms_key entries appended after their IDs are known.


def _atexit_cleanup() -> None:
    try:
        run_cleanup(CLEANUP_MANIFEST)
    except Exception:
        pass


atexit.register(_atexit_cleanup)


def scan_for_orphans() -> list[str]:
    client = boto3.client(
        "resourcegroupstaggingapi",
        aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
        aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
        aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
        region_name=REGION,
    )
    arns: list[str] = []
    paginator = client.get_paginator("get_resources")
    for page in paginator.paginate(
        TagFilters=[{"Key": LAB_TAG_KEY, "Values": [LAB_TAG_VALUE]}],
    ):
        for item in page.get("ResourceTagMappingList", []):
            arns.append(item.get("ResourceARN", ""))
    return arns


orphans = scan_for_orphans()
if orphans:
    kms_client = boto3.client(
        "kms",
        aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
        aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
        aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
        region_name=REGION,
    )
    active_orphans: list[str] = []
    for arn in orphans:
        if ":kms:" in arn and ":key/" in arn:
            try:
                desc = kms_client.describe_key(KeyId=arn)
                state = desc.get("KeyMetadata", {}).get("KeyState")
                if state == "PendingDeletion":
                    continue
            except ClientError:
                continue
        active_orphans.append(arn)
    if active_orphans:
        print(f"BLOCKED: existing resources tagged labrun:lab-slug={LAB_TAG_VALUE} were found:")
        for arn in active_orphans:
            print("  -", arn)
        print()
        print("Run the cleanup cell at the bottom of this notebook first, then re-run setup.")
        raise SystemExit(1)

print("No active prior orphans found. Safe to create resources for this session.")

## Task 1: Create the owner + partner roles, the CMK with bucket-default encryption, and the bucket policy scoped via aws:PrincipalArn

The lab simulates a cross-account partner using two roles in the same AWS account. The owner role represents the bucket-owning account; the partner role represents the third-party integration. The bucket-policy and key-grant shapes are identical to the real cross-account form (the only difference is the `Principal.AWS` ARN format).

Build it in this order:

1. Create the owner role and partner role with the account-root trust policy. The partner role gets `s3:GetObject` on the bucket prefix as an inline policy so the request reaches S3 authorization.
2. Create the CMK with an explicit key policy granting the caller admin (the grant we add in Task 2 needs CMK admin permissions to even create).
3. Create the alias.
4. Create the bucket with SSE-KMS default encryption referencing the CMK ARN.
5. Attach a bucket policy with one `Allow` statement that names the partner role ARN as `Principal.AWS`, allows `s3:GetObject` on `arn:aws:s3:::{bucket}/reports/*`, and pins the request principal via `Condition.StringEquals.aws:PrincipalArn`.
6. Upload the seed object.

In [ ]:
# NBVAL_SKIP
# Task 1: roles, CMK, alias, bucket with SSE-KMS default, bucket policy, seed object.

iam = boto3.client(
    "iam",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
kms = boto3.client(
    "kms",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
s3 = boto3.client(
    "s3",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

trust_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {"AWS": f"arn:aws:iam::{ACCOUNT_ID}:root"},
            "Action": "sts:AssumeRole",
        }
    ],
}
for role_name in (OWNER_ROLE_NAME, PARTNER_ROLE_NAME):
    try:
        iam.create_role(
            RoleName=role_name,
            AssumeRolePolicyDocument=json.dumps(trust_policy),
            Description=f"labrun lab role {role_name}",
            Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
        )
        print(f"Created role: {role_name}")
    except ClientError as e:
        if e.response["Error"]["Code"] == "EntityAlreadyExists":
            print(f"Role {role_name} already exists, reusing.")
        else:
            raise

PARTNER_ROLE_ARN = f"arn:aws:iam::{ACCOUNT_ID}:role/{PARTNER_ROLE_NAME}"
OWNER_ROLE_ARN = f"arn:aws:iam::{ACCOUNT_ID}:role/{OWNER_ROLE_NAME}"

# Partner-role inline policy: S3 GetObject on the reports prefix. In a real
# cross-account setup this policy lives in the partner's own AWS account; the
# lab simulation co-locates it.
partner_s3_inline = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": "s3:GetObject",
            "Resource": f"arn:aws:s3:::{BUCKET_NAME}/reports/*",
        }
    ],
}
iam.put_role_policy(
    RoleName=PARTNER_ROLE_NAME,
    PolicyName=f"labrun-{PARTNER_ROLE_NAME}-s3-read",
    PolicyDocument=json.dumps(partner_s3_inline),
)

# CMK with admin clause for the lab caller and account root.
key_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "LabCallerAdmin",
            "Effect": "Allow",
            "Principal": {"AWS": CALLER_ARN},
            "Action": "kms:*",
            "Resource": "*",
        },
        {
            "Sid": "AccountRootAdmin",
            "Effect": "Allow",
            "Principal": {"AWS": f"arn:aws:iam::{ACCOUNT_ID}:root"},
            "Action": "kms:*",
            "Resource": "*",
        },
    ],
}
create_resp = kms.create_key(
    Description="labrun lab CMK",
    KeyUsage="ENCRYPT_DECRYPT",
    Policy=json.dumps(key_policy),
    Tags=[{"TagKey": LAB_TAG_KEY, "TagValue": LAB_TAG_VALUE}],
)
CMK_KEY_ID = create_resp["KeyMetadata"]["KeyId"]
CMK_KEY_ARN = create_resp["KeyMetadata"]["Arn"]
print(f"CMK created: {CMK_KEY_ID}")

kms.create_alias(AliasName=CMK_ALIAS, TargetKeyId=CMK_KEY_ID)

CLEANUP_MANIFEST.append(
    CleanupResource(
        type="kms_key",
        id=CMK_KEY_ID,
        region=REGION,
        cli_delete_command=(
            f"aws kms schedule-key-deletion --key-id {CMK_KEY_ID} --pending-window-in-days 7"
        ),
    )
)

if REGION == "us-east-1":
    s3.create_bucket(Bucket=BUCKET_NAME)
else:
    s3.create_bucket(
        Bucket=BUCKET_NAME,
        CreateBucketConfiguration={"LocationConstraint": REGION},
    )
s3.put_bucket_tagging(
    Bucket=BUCKET_NAME,
    Tagging={"TagSet": [{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}]},
)
s3.put_bucket_encryption(
    Bucket=BUCKET_NAME,
    ServerSideEncryptionConfiguration={
        "Rules": [
            {
                "ApplyServerSideEncryptionByDefault": {
                    "SSEAlgorithm": "aws:kms",
                    "KMSMasterKeyID": CMK_KEY_ARN,
                },
                "BucketKeyEnabled": True,
            }
        ]
    },
)
print(f"Bucket created with SSE-KMS default: {BUCKET_NAME}")

# Bucket policy: one Allow statement that names the partner role ARN as
# Principal.AWS and pins the request principal via aws:PrincipalArn.
bucket_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "AllowPartnerRead",
            "Effect": "Allow",
            "Principal": {"AWS": f"arn:aws:iam::{ACCOUNT_ID}:root"},
            "Action": "s3:GetObject",
            "Resource": f"arn:aws:s3:::{BUCKET_NAME}/reports/*",
            "Condition": {
                "StringEquals": {"aws:PrincipalArn": PARTNER_ROLE_ARN}
            },
        }
    ],
}
# YOUR CODE: Attach the bucket policy using s3.put_bucket_policy() with
# Bucket=BUCKET_NAME and Policy=json.dumps(bucket_policy).

s3.put_object(Bucket=BUCKET_NAME, Key=SEED_KEY, Body=b"Q4 partner report extract\n")
print(f"Seed object uploaded: s3://{BUCKET_NAME}/{SEED_KEY}")
print("Waiting 10 seconds for IAM and S3 propagation...")
time.sleep(10)
print("Done.")

In [ ]:
# NBVAL_SKIP
# Checkpoint 1: Bucket policy allows partner role GetObject scoped via
# aws:PrincipalArn. Accepts either the partner-role ARN or the account root
# ARN as Principal.AWS, as long as the Condition pins aws:PrincipalArn to
# the partner role ARN.

def checkpoint_1(session):
    from labrun_checks import CheckpointResult
    try:
        s3_client = boto3.client(
            "s3",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        try:
            pol = s3_client.get_bucket_policy(Bucket=BUCKET_NAME)
        except ClientError as e:
            if e.response["Error"]["Code"] == "NoSuchBucketPolicy":
                return CheckpointResult(
                    status="fail",
                    error_reason=f"Bucket {BUCKET_NAME} has no bucket policy attached.",
                )
            return CheckpointResult(status="error", error_reason=str(e))

        policy_doc = json.loads(pol["Policy"])
        statements = policy_doc.get("Statement", [])
        partner_role_arn = f"arn:aws:iam::{ACCOUNT_ID}:role/{PARTNER_ROLE_NAME}"
        account_root_arn = f"arn:aws:iam::{ACCOUNT_ID}:root"
        expected_resource = f"arn:aws:s3:::{BUCKET_NAME}/reports/*"
        action_wildcards = {"s3:*", "*"}

        for stmt in statements:
            if stmt.get("Effect") != "Allow":
                continue
            p = stmt.get("Principal", {})
            aws_principal = p.get("AWS")
            if isinstance(aws_principal, str):
                principals = [aws_principal]
            elif isinstance(aws_principal, list):
                principals = aws_principal
            else:
                principals = []
            if partner_role_arn not in principals and account_root_arn not in principals:
                continue
            actions = stmt.get("Action", [])
            if isinstance(actions, str):
                actions = [actions]
            action_set = set(actions)
            if "s3:GetObject" not in action_set and not (action_set & action_wildcards):
                continue
            resources = stmt.get("Resource", [])
            if isinstance(resources, str):
                resources = [resources]
            if expected_resource not in resources:
                continue
            cond = stmt.get("Condition", {})
            str_eq = cond.get("StringEquals", {})
            principal_arn_cond = str_eq.get("aws:PrincipalArn")
            if isinstance(principal_arn_cond, str):
                cond_values = [principal_arn_cond]
            elif isinstance(principal_arn_cond, list):
                cond_values = principal_arn_cond
            else:
                cond_values = []
            # If Principal is the partner role ARN directly, the Condition is
            # not strictly required by AWS; the lab pattern uses account root +
            # Condition for stability, but accept either shape.
            if partner_role_arn in principals or partner_role_arn in cond_values:
                return CheckpointResult(status="pass")

        return CheckpointResult(
            status="fail",
            error_reason=(
                f"Bucket policy is missing an Allow statement for s3:GetObject on "
                f"{expected_resource!r} that names {partner_role_arn!r} (either as "
                f"Principal.AWS directly or as Condition.StringEquals.aws:PrincipalArn)."
            ),
        )
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(1, checkpoint_1)

<details><summary>Hint 1 (nudge)</summary>

The checkpoint walks the bucket policy looking for an Allow statement that names the partner role (directly or via PrincipalArn Condition), allows GetObject, and lists the reports prefix as Resource. The failure message tells you which piece is missing.

</details>

<details><summary>Hint 2 (direction)</summary>

Two valid shapes. Either `Principal.AWS = PARTNER_ROLE_ARN` directly (simple form), or `Principal.AWS = account root ARN` plus `Condition.StringEquals.aws:PrincipalArn = PARTNER_ROLE_ARN` (more durable; survives partner role recreation). The lab uses the second shape. In either case, Action is `s3:GetObject`, Resource is `arn:aws:s3:::{bucket}/reports/*`.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for the YOUR CODE blank in Task 1:

```python
s3.put_bucket_policy(Bucket=BUCKET_NAME, Policy=json.dumps(bucket_policy))
```

The bucket_policy dict that ships in the cell uses the account-root-plus-PrincipalArn-Condition pattern; just attach it.

</details>

**Wallet check.** Still fractions of a penny. Bucket-policy attach, IAM role create, and the CMK existence prorated over a few minutes all round to less than $0.01.

## Task 2: Create a KMS grant giving the partner role Decrypt and DescribeKey

The key policy could grant the partner role Decrypt directly, but a KMS grant is the AWS-designed mechanism for revocable, scoped delegation that does not require admin-on-the-CMK to manage and shows up in `kms.list_grants` for security audit independent of the key policy.

Build it in this order:

1. Call `kms.create_grant` with `KeyId=CMK_KEY_ID`, `GranteePrincipal=PARTNER_ROLE_ARN`, and `Operations=["Decrypt", "DescribeKey"]`.
2. Capture the returned `GrantId` and store it in `GRANT_ID`.
3. Append the grant to `CLEANUP_MANIFEST` so cleanup revokes it before scheduling the key for deletion.

In [ ]:
# NBVAL_SKIP
# Task 2: Create the KMS grant and register it for cleanup.

kms = boto3.client(
    "kms",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

PARTNER_ROLE_ARN = f"arn:aws:iam::{ACCOUNT_ID}:role/{PARTNER_ROLE_NAME}"

# YOUR CODE: Call kms.create_grant() with KeyId=CMK_KEY_ID,
# GranteePrincipal=PARTNER_ROLE_ARN, Operations=["Decrypt", "DescribeKey"],
# and Name="labrun-partner-decrypt". Store the response in grant_resp.

GRANT_ID = grant_resp["GrantId"]
print(f"Grant created: {GRANT_ID}")

CLEANUP_MANIFEST.insert(
    0,
    CleanupResource(
        type="kms_grant",
        id=GRANT_ID,
        parent=CMK_KEY_ID,
        region=REGION,
        cli_delete_command=(
            f"aws kms revoke-grant --key-id {CMK_KEY_ID} --grant-id {GRANT_ID}"
        ),
    ),
)
print("Grant registered in CLEANUP_MANIFEST.")

In [ ]:
# NBVAL_SKIP
# Checkpoint 2: KMS grant exists on the CMK with the partner role as
# GranteePrincipal and Operations covering Decrypt + DescribeKey.

def checkpoint_2(session):
    from labrun_checks import CheckpointResult
    try:
        kms_client = boto3.client(
            "kms",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        if not CMK_KEY_ID:
            return CheckpointResult(
                status="fail",
                error_reason="CMK_KEY_ID is not set. Run Task 1 to create the CMK.",
            )
        partner_role_arn = f"arn:aws:iam::{ACCOUNT_ID}:role/{PARTNER_ROLE_NAME}"
        required_ops = {"Decrypt", "DescribeKey"}
        op_wildcards = {"*", "kms:*"}

        paginator = kms_client.get_paginator("list_grants")
        for page in paginator.paginate(KeyId=CMK_KEY_ID):
            for grant in page.get("Grants", []):
                if grant.get("GranteePrincipal") != partner_role_arn:
                    continue
                ops = set(grant.get("Operations", []))
                if required_ops.issubset(ops) or bool(ops & op_wildcards):
                    return CheckpointResult(status="pass")

        return CheckpointResult(
            status="fail",
            error_reason=(
                f"No KMS grant on key {CMK_KEY_ID} names {partner_role_arn!r} as "
                f"GranteePrincipal with Decrypt + DescribeKey operations (or wildcard)."
            ),
        )
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(2, checkpoint_2)

<details><summary>Hint 1 (nudge)</summary>

If the checkpoint cannot find the grant, the `create_grant` call did not run or it referenced the wrong key or principal. Make sure GRANT_ID is set before you proceed.

</details>

<details><summary>Hint 2 (direction)</summary>

`kms.create_grant(KeyId=CMK_KEY_ID, GranteePrincipal=PARTNER_ROLE_ARN, Operations=["Decrypt", "DescribeKey"], Name="labrun-partner-decrypt")`. The `Operations` list must contain at least both names; `Encrypt` is optional for this lab.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 2:

```python
grant_resp = kms.create_grant(
    KeyId=CMK_KEY_ID,
    GranteePrincipal=PARTNER_ROLE_ARN,
    Operations=["Decrypt", "DescribeKey"],
    Name="labrun-partner-decrypt",
)
GRANT_ID = grant_resp["GrantId"]
```

</details>

**Wallet check.** KMS grants do not move the wallet. `create_grant`, `list_grants`, and `revoke_grant` are all free control-plane operations on the CMK.

## Task 3: Partner role reads the encrypted object end to end via the grant

Bucket policy allows the partner role at the S3 layer; the KMS grant allows the partner role at the KMS layer. Both must succeed for GetObject to return plaintext.

Build it in this order:

1. Assume the partner role.
2. Build a fresh S3 client from the temp credentials (with `aws_session_token`).
3. Call `get_object` on `reports/seed.txt` and confirm the response carries `ServerSideEncryption=aws:kms` and `SSEKMSKeyId` matching the lab CMK ARN.

In [ ]:
# NBVAL_SKIP
# Task 3: Assume the partner role and read the seed object.

sts = boto3.client(
    "sts",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

PARTNER_ROLE_ARN = f"arn:aws:iam::{ACCOUNT_ID}:role/{PARTNER_ROLE_NAME}"

# YOUR CODE: Call sts.assume_role(RoleArn=PARTNER_ROLE_ARN,
# RoleSessionName="labrun-partner-read") and store the response in
# assume_resp.

temp_creds = assume_resp["Credentials"]

# YOUR CODE: Build a fresh boto3 S3 client from temp_creds with
# aws_session_token=temp_creds["SessionToken"] and assign it to s3_partner.

resp = s3_partner.get_object(Bucket=BUCKET_NAME, Key=SEED_KEY)
print("GetObject succeeded as partner role via the grant-backed Decrypt path.")
print(f"  ServerSideEncryption: {resp.get('ServerSideEncryption')}")
print(f"  SSEKMSKeyId:          {resp.get('SSEKMSKeyId')}")
body = resp["Body"].read()
print(f"  Body bytes:           {body!r}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 3: Partner role reads the seed object via the grant-backed
# Decrypt path. Response carries SSE-KMS metadata pointing at the lab CMK.

def checkpoint_3(session):
    from labrun_checks import CheckpointResult
    try:
        sts_client = boto3.client(
            "sts",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        role_arn = f"arn:aws:iam::{ACCOUNT_ID}:role/{PARTNER_ROLE_NAME}"

        last_error = None
        delay = 2
        elapsed = 0
        assume_resp = None
        while elapsed < 60:
            try:
                assume_resp = sts_client.assume_role(
                    RoleArn=role_arn,
                    RoleSessionName="labrun-checkpoint-3",
                )
                last_error = None
                break
            except ClientError as e:
                last_error = e
                time.sleep(delay)
                elapsed += delay
                delay = min(delay * 2, 8)
        if last_error is not None or assume_resp is None:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Could not assume {role_arn} within 60s. "
                    f"{repr(last_error) if last_error else 'no response'}"
                ),
            )

        temp_creds = assume_resp["Credentials"]
        s3_assumed = boto3.client(
            "s3",
            aws_access_key_id=temp_creds["AccessKeyId"],
            aws_secret_access_key=temp_creds["SecretAccessKey"],
            aws_session_token=temp_creds["SessionToken"],
            region_name=REGION,
        )
        try:
            resp = s3_assumed.get_object(Bucket=BUCKET_NAME, Key=SEED_KEY)
        except ClientError as e:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Partner role GetObject failed: {e.response['Error']['Code']}: "
                    f"{e.response['Error']['Message']}. Verify the bucket policy clause "
                    f"AND the KMS grant are both in place."
                ),
            )
        if resp.get("ServerSideEncryption") != "aws:kms":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Response ServerSideEncryption is {resp.get('ServerSideEncryption')!r}, "
                    f"expected 'aws:kms'."
                ),
            )
        sse_key = resp.get("SSEKMSKeyId", "")
        if CMK_KEY_ID not in sse_key and (CMK_KEY_ARN or "") != sse_key:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Response SSEKMSKeyId is {sse_key!r}, expected to reference "
                    f"the lab CMK ({CMK_KEY_ARN})."
                ),
            )
        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(3, checkpoint_3)

<details><summary>Hint 1 (nudge)</summary>

Two failure shapes are possible. Either assume_role itself fails (trust policy or IAM propagation), or assume_role succeeds and GetObject fails (bucket policy clause missing or KMS grant missing).

</details>

<details><summary>Hint 2 (direction)</summary>

Build a NEW boto3 S3 client from the temp credentials with `aws_session_token` set. If GetObject fails with AccessDenied that references KMS, the grant is missing or the wrong key id. If it fails with AccessDenied that references S3, the bucket policy clause is missing or the PrincipalArn condition does not match the partner role.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 3:

```python
assume_resp = sts.assume_role(
    RoleArn=PARTNER_ROLE_ARN,
    RoleSessionName="labrun-partner-read",
)
temp_creds = assume_resp["Credentials"]

s3_partner = boto3.client(
    "s3",
    aws_access_key_id=temp_creds["AccessKeyId"],
    aws_secret_access_key=temp_creds["SecretAccessKey"],
    aws_session_token=temp_creds["SessionToken"],
    region_name=REGION,
)
```

</details>

**Wallet check.** Still well under one cent in-session: STS AssumeRole is free, GetObject is a fraction of a penny, and the Decrypt call falls inside the 20k monthly free tier.

## Task 4: Revoke the bucket policy clause and prove the same GetObject now fails AccessDenied

The KMS grant alone is not enough. S3 authorizes the request first; only if S3 says yes does the call proceed to KMS Decrypt. Remove the bucket policy clause and the partner role gets AccessDenied at the S3 layer, even though the KMS grant is still in place.

Build it in this order:

1. Delete the bucket policy entirely (the simplest form of revocation; an empty-statement policy works too).
2. Assume the partner role again.
3. Build a fresh S3 client from the temp credentials.
4. Call `get_object` on the same key and assert it raises `ClientError` with `Error.Code == "AccessDenied"`.

In [ ]:
# NBVAL_SKIP
# Task 4: Revoke the bucket policy, re-assume the partner role, and prove
# AccessDenied at the S3 layer (grant is still in place).

s3 = boto3.client(
    "s3",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
sts = boto3.client(
    "sts",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

# YOUR CODE: Remove the bucket policy with s3.delete_bucket_policy(Bucket=BUCKET_NAME).

print("Bucket policy removed.")
print("Waiting 5 seconds for S3 propagation...")
time.sleep(5)

PARTNER_ROLE_ARN = f"arn:aws:iam::{ACCOUNT_ID}:role/{PARTNER_ROLE_NAME}"
assume_resp = sts.assume_role(
    RoleArn=PARTNER_ROLE_ARN,
    RoleSessionName="labrun-partner-revoked",
)
temp_creds = assume_resp["Credentials"]

s3_partner = boto3.client(
    "s3",
    aws_access_key_id=temp_creds["AccessKeyId"],
    aws_secret_access_key=temp_creds["SecretAccessKey"],
    aws_session_token=temp_creds["SessionToken"],
    region_name=REGION,
)

try:
    s3_partner.get_object(Bucket=BUCKET_NAME, Key=SEED_KEY)
    print("UNEXPECTED: partner role still read the object. Bucket policy removal did not stick.")
except ClientError as e:
    code_str = e.response["Error"]["Code"]
    print(f"Denied as expected: {code_str}")
    print(f"  Full message: {e.response['Error']['Message']}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 4: Bucket policy is gone (or no longer grants partner access);
# partner role gets AccessDenied on GetObject for the same object; KMS grant
# is still intact (the failure is at the S3 layer).

def checkpoint_4(session):
    from labrun_checks import CheckpointResult
    try:
        s3_client = boto3.client(
            "s3",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        partner_role_arn = f"arn:aws:iam::{ACCOUNT_ID}:role/{PARTNER_ROLE_NAME}"

        # Confirm bucket policy no longer grants the partner role.
        try:
            pol = s3_client.get_bucket_policy(Bucket=BUCKET_NAME)
            policy_doc = json.loads(pol["Policy"])
            for stmt in policy_doc.get("Statement", []):
                if stmt.get("Effect") != "Allow":
                    continue
                p = stmt.get("Principal", {})
                aws_principal = p.get("AWS")
                principals = (
                    [aws_principal] if isinstance(aws_principal, str) else (aws_principal or [])
                )
                cond = stmt.get("Condition", {})
                str_eq = cond.get("StringEquals", {})
                principal_arn_cond = str_eq.get("aws:PrincipalArn")
                cond_values = (
                    [principal_arn_cond]
                    if isinstance(principal_arn_cond, str)
                    else (principal_arn_cond or [])
                )
                if partner_role_arn in principals or partner_role_arn in cond_values:
                    return CheckpointResult(
                        status="fail",
                        error_reason=(
                            f"Bucket policy still grants {partner_role_arn!r}. Task 4 must "
                            f"remove the Allow clause (delete_bucket_policy or rewrite without it)."
                        ),
                    )
        except ClientError as e:
            if e.response["Error"]["Code"] != "NoSuchBucketPolicy":
                return CheckpointResult(status="error", error_reason=str(e))

        # Independently assume the partner role and prove the read fails.
        sts_client = boto3.client(
            "sts",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        assume_resp = sts_client.assume_role(
            RoleArn=partner_role_arn,
            RoleSessionName="labrun-checkpoint-4",
        )
        temp_creds = assume_resp["Credentials"]
        s3_assumed = boto3.client(
            "s3",
            aws_access_key_id=temp_creds["AccessKeyId"],
            aws_secret_access_key=temp_creds["SecretAccessKey"],
            aws_session_token=temp_creds["SessionToken"],
            region_name=REGION,
        )
        try:
            s3_assumed.get_object(Bucket=BUCKET_NAME, Key=SEED_KEY)
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Partner role unexpectedly read the object after the bucket policy was "
                    f"removed. The S3 layer should refuse the request."
                ),
            )
        except ClientError as e:
            code_str = e.response["Error"]["Code"]
            if code_str != "AccessDenied":
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"Expected AccessDenied, got {code_str}. {e.response['Error']['Message']}"
                    ),
                )
        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(4, checkpoint_4)

<details><summary>Hint 1 (nudge)</summary>

If the partner read succeeded, the bucket policy still grants access. If the error code is something other than AccessDenied, the failure is happening at a different layer than expected.

</details>

<details><summary>Hint 2 (direction)</summary>

Simplest revocation: `s3.delete_bucket_policy(Bucket=BUCKET_NAME)`. You can also call `put_bucket_policy` with a statement that omits the partner Allow clause; both shapes pass the checkpoint. The KMS grant stays in place because the lab's pedagogical point is that both layers must allow for the request to succeed.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for the YOUR CODE blank in Task 4:

```python
s3.delete_bucket_policy(Bucket=BUCKET_NAME)
```

</details>

**Wallet check.** Still under one cent in-session. `delete_bucket_policy` is free, one denied GetObject is a fraction of a penny, and the KMS Decrypt that fires before the S3 denial is inside your 20k monthly free tier.

## Cleanup

Tear it all down. The cell below revokes the KMS grant, deletes the bucket, both roles, the CMK alias, and schedules the CMK for deletion. Running cleanup twice is safe.

In [ ]:
# NBVAL_SKIP
# Cleanup. Tear down every resource in CLEANUP_MANIFEST.

import sys

result = run_cleanup(CLEANUP_MANIFEST)

for warning in result.warnings:
    print(warning)
for orphan in result.orphans:
    print(orphan)
if result.warnings or result.orphans:
    print()

failed_ids: set[str] = set()
for w in result.warnings:
    for r in CLEANUP_MANIFEST:
        if r.id in w:
            failed_ids.add(r.id)
            break

critical_total = 0
standard_total = len(CLEANUP_MANIFEST)
critical_destroyed = critical_total
standard_destroyed = standard_total - len(failed_ids)
failed_count = len(failed_ids)

print("Cleanup complete.")
print(f"Critical resources destroyed: {critical_destroyed}")
print(f"Standard resources destroyed: {standard_destroyed}")
print(f"Resources that failed to delete: {failed_count} (see above for CLI commands)")
print("If K > 0, your AWS account may still incur charges. Resolve before closing this notebook.")

cleanup(status=result.status)

if failed_count > 0:
    sys.exit(1)

**Session total: under $0.05 in-session, plus approximately $0.23 residual per CMK over the 7-day scheduled-deletion window.**

## Reflection

These are not graded. They are for you.

1. Why does AWS require BOTH a bucket-policy clause AND a KMS authorization (grant or key-policy clause) to share an SSE-KMS-encrypted object? Couldn't a single grant on the CMK be enough, given that the partner cannot decrypt without it?

2. Walk through every authorization surface AWS evaluates when the partner role calls GetObject on the encrypted seed object: the partner role's IAM, the bucket policy, the bucket-owner ACL, the key policy, and any KMS grants. List each in evaluation order and what must permit the call.

## Exam-Style Practice

**Question 1 (Domain 1, cross-account access pattern selection):**

A partner team in a different AWS account needs durable, audited read access to a curated S3 prefix in your account for an integration that will run nightly for the next two years. Which access pattern best fits the requirement?

A. Generate a presigned URL each night and email it to the partner before their batch runs.

B. Make the bucket public for the prefix and rely on object obscurity for security.

C. Add a bucket policy clause allowing GetObject scoped via `aws:PrincipalArn` to the partner role ARN, and have the partner's IAM grant the role `s3:GetObject` on your bucket ARN.

D. Add the partner account ID to a bucket ACL grantee with READ permission.

<details><summary>Show answer</summary>

**Correct: C.**

- A is wrong: presigned URLs are short-lived and require an operational handoff per batch. A two-year nightly integration is exactly the long-lived case where you want a policy-driven grant, not a presigned URL.
- B is wrong: making a bucket public is a security incident, not an access pattern. AWS Block Public Access exists to prevent this and the partner integration runs in their own AWS account; they do not need anonymous access.
- C is correct: bucket-policy Allow + Condition on `aws:PrincipalArn` + reciprocal IAM grant in the partner account is the AWS-documented durable cross-account pattern. Both sides must grant for the request to succeed; the audit trail names the partner principal on every call.
- D is wrong: bucket ACLs are deprecated for new use cases. AWS strongly recommends bucket policies plus IAM. ACLs also do not support fine-grained conditions like PrincipalArn.

</details>

**Question 2 (Domain 1, KMS grant vs key policy clause):**

You own a Customer Managed KMS key and need to allow a role in another AWS account to call `Decrypt` against it. The delegation should be revocable at any time without modifying the key policy and should appear in a CMK-side audit list that the security team reviews monthly. Which mechanism best fits?

A. Add a Statement to the CMK's key policy with `Principal.AWS` set to the cross-account role ARN and `Action: kms:Decrypt`. Revoke by editing the policy.

B. Create a KMS grant on the CMK with `GranteePrincipal` set to the cross-account role and `Operations` set to `Decrypt`. Revoke with `kms.revoke_grant`.

C. Attach an IAM policy to the cross-account role granting `kms:Decrypt` on the CMK ARN. Revoke by detaching the policy.

D. Add the cross-account role to the CMK's `via-service` Condition for `kms.amazonaws.com`. Revoke by removing it.

<details><summary>Show answer</summary>

**Correct: B.**

- A works mechanically but modifying the key policy requires admin permissions on the CMK each time you grant or revoke. The reviewer audits the key policy, not a grant list; revocation looks like a policy edit, which is heavier than the question's revocability requirement.
- B is correct: KMS grants are the AWS-designed mechanism for cross-account, scoped, revocable delegation that the security team audits via `list_grants` separately from the key policy. `revoke_grant` is the explicit single-API-call revocation.
- C does not work alone for cross-account: the partner role's IAM policy can grant Decrypt, but the CMK side must ALSO permit the cross-account principal via key policy or grant. IAM in the partner account is not enough to authorize Decrypt on a CMK in another account.
- D is wrong: `kms:via-service` is a Condition that scopes the calling AWS service (e.g., S3, EBS), not a way to add principals.

</details>